# Simulate TMS-fMRI Sessions with V2 Parameters
## Control: REST Initialization for Both Conditions (Amplitude-Only Comparison)

**Version 2 Key Changes:**
- **BURN_IN:** 30 (was 10)
- **NOISE_SIGMA:** 0.28 (was 0.45)
- **STIM_AMP:** 10.0 (was 100.0)
- **RHO_MM:** 60.0 (was 50.0)
- **Control Design:** Both REST and STIM(amp=0) use REST empirical initialization
  - This isolates the amplitude effect from initialization effects
  - True amplitude-only comparison: same brain state, different amplitude

**Validation:** Includes full FC analysis + comparative plots of STIM_AMP vs STIM_AMP=0

<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Simulate_TMS_fMRI_ANN2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Simulate_TMS_fMRI_ANN2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulate TMS-fMRI Sessions with Subject-Specific, Stimulus-Agnostic ANN Models

**Objective:** Use subject-specific models trained on TMS data (without explicit stimulus encoding) to simulate both resting-state and stimulation sessions with empirical TMS timing and target location information.

**Key Innovation:** Unlike stimulus-aware models, these models capture intrinsic brain dynamics learned from recordings during stimulation. When we apply realistic TMS timing and spatial location during simulation, we test whether learned dynamics naturally respond to stimulation constraints.

**Workflow:**
1. Load subject-specific stimulus-agnostic MLP models trained on concatenated TMS sessions
2. Load empirical dataset with rest and stim sessions, including TMS target regions and timing
3. For each subject, generate synthetic sessions:
   - Rest: Simulate resting-state activity without stimulation
   - Stim: Simulate brain response to empirical TMS timing and target location using spatial Gaussian kernel
4. Save simulated dataset to preprocessed_subjects_tms_fmri
5. Validate simulations match empirical functional connectivity

## Step 1: Setup and Configuration

In [ ]:
# --- Setup ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys, pickle, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Clone repo + add to path
repo_dir = "/content/BrainStim_ANN_fMRI_HCP"
if not os.path.exists(repo_dir):
    !git clone https://github.com/grabuffo/BrainStim_ANN_fMRI_HCP.git
else:
    print("Repo already exists ✅")

sys.path.append(repo_dir)
from src.preprocessing_tms_fmri import preprocess_run
from src.NPI import build_model, device

print(f"PyTorch device: {device}")

In [ ]:
# --- Configuration Parameters ---
ROI_num = 450
using_steps = 3
tr_rest = 2.0
tr_stim = 2.4
remove_initial_trs = 12  # Matching new model training (not 30)
low_hz, high_hz = 0.008, 0.08

# Simulation parameters (NEW VALUES)
BURN_IN = 30
NOISE_SIGMA = 0.28
STIM_AMP = 10.0
STIM_DURATION_S = tr_stim
RHO_MM = 60.0  # Gaussian spread (mm) for spatial TMS effect

rng = np.random.default_rng(42)

# Paths
base_dir = "/content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data"
data_dir = os.path.join(base_dir, "TMS_fMRI")
preproc_dir = os.path.join(base_dir, "preprocessed_subjects_tms_fmri")
dataset_pkl = os.path.join(data_dir, "dataset_tian50_schaefer400_allruns.pkl")

# Load models from NEW training workflow
weights_dir = os.path.join(preproc_dir, "trained_models_MLP")  # New stimulus-agnostic models

# Output directory for simulated dataset
out_dir = os.path.join(preproc_dir, "simulated_tms_fmri_stim_info_v2")
os.makedirs(out_dir, exist_ok=True)

out_pkl = os.path.join(out_dir, "dataset_simulated_with_stim_info.pkl")
results_json = os.path.join(out_dir, "simulation_summary.json")

print(f"Base dir: {base_dir}")
print(f"Weights dir: {weights_dir}")
print(f"Output dir: {out_dir}")
print(f"\nConfiguration:")
print(f"  ROI_num: {ROI_num}, using_steps: {using_steps}")
print(f"  TR_rest: {tr_rest}s, TR_stim: {tr_stim}s")
print(f"  remove_initial_trs: {remove_initial_trs}")
print(f"  BURN_IN: {BURN_IN}, NOISE_SIGMA: {NOISE_SIGMA}, STIM_AMP: {STIM_AMP}, RHO_MM: {RHO_MM}mm")

## Step 2: Load Distance Matrix and Spatial Gaussian Kernel

In [ ]:
# --- Load distance matrix for spatial Gaussian kernel ---
try:
    dist_path = os.path.join(data_dir, "atlases", "distance_matrix_450x450_Tian50_Schaefer400.npy")
    D = np.load(dist_path)
    W = np.exp(-(D ** 2) / (2.0 * (RHO_MM ** 2))).astype(np.float32)
    W /= (W[np.arange(ROI_num), np.arange(ROI_num)][:, None] + 1e-8)  # Normalize so target = 1
    print(f"✓ Loaded distance matrix: W shape {W.shape}, range [{W.min():.4f}, {W.max():.4f}]")
    print(f"  Gaussian spread: RHO_MM = {RHO_MM} mm")
    print(f"  Diagonal values (self-weights): {W.diagonal()[:5]}...")
except FileNotFoundError as e:
    print(f"⚠ Distance matrix not found: {e}")
    print(f"  Will use direct stimulation only (no spatial spread)")
    W = None

## Step 3: Define Model Architecture and Load Trained Models

In [ ]:
# --- Model Architecture: Stimulus-Agnostic MLP ---
# Using NPI helper function instead of defining class manually
# Model architecture:
#   Input: Brain state only (using_steps * ROI_num dimensions = 3 * 450 = 1350)
#   Hidden layers: 256 → 128 hidden units with ReLU activations
#   Output: Next predicted ROI activation (450 dims)
# Key difference from old models: NO stimulus encoding. Just intrinsic dynamics.

print("✓ Will instantiate models using src.NPI.build_model()")

In [ ]:
# --- Load subject-specific trained models ---
print("Loading subject-specific stimulus-agnostic models...\n")

trained_models = {}
model_pattern = '_MLP.pt'  # New models save as sub-ID_MLP.pt

# List available model files
if os.path.exists(weights_dir):
    model_files = [f for f in os.listdir(weights_dir) if f.endswith(model_pattern)]
    print(f"Found {len(model_files)} model files in {weights_dir}")
else:
    print(f"⚠ Weights directory not found: {weights_dir}")
    model_files = []

for model_file in sorted(model_files):
    sub_id = model_file.replace(model_pattern, '')
    model_path = os.path.join(weights_dir, model_file)

    try:
        # Load the checkpoint
        torch.serialization.add_safe_globals([torch.nn.modules.linear.Linear, torch.nn.modules.activation.ReLU])
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)

        # Handle both cases: saved as state_dict or as full model
        if isinstance(checkpoint, dict):
            # Case 1: Saved as state_dict
            model = build_model("MLP", ROI_num=ROI_num, using_steps=using_steps).to(device)
            model.load_state_dict(checkpoint)
        else:
            # Case 2: Saved as full model object
            model = checkpoint.to(device) if hasattr(checkpoint, 'to') else checkpoint

        model.eval()
        trained_models[sub_id] = model
        print(f"✓ {sub_id}")
    except Exception as e:
        print(f"✗ {sub_id} - skipping ({str(e)[:50]}...)")

print(f"\n✅ Loaded {len(trained_models)} models ready for simulation")

## Step 4: Load Empirical Dataset and Verify Subject Matching

In [ ]:
# --- Load empirical dataset ---
print(f"Loading empirical dataset from {dataset_pkl}...")
with open(dataset_pkl, "rb") as f:
    dataset_emp = pickle.load(f)

print(f"✓ Loaded {len(dataset_emp)} subjects")

# Verify dataset structure
sample_sub = list(dataset_emp.keys())[0]
print(f"\nSample subject: {sample_sub}")
print(f"  Available conditions: {list(dataset_emp[sample_sub].keys())}")
if 'task-rest' in dataset_emp[sample_sub]:
    sample_rest = dataset_emp[sample_sub]['task-rest']
    print(f"  Rest runs: {list(sample_rest.keys())[:3]}...")
    first_rest_run = list(sample_rest.values())[0]
    print(f"  Rest data keys: {list(first_rest_run.keys())}")
    if 'time series' in first_rest_run:
        print(f"  Rest TS shape: {first_rest_run['time series'].shape}")

if 'task-stim' in dataset_emp[sample_sub]:
    sample_stim = dataset_emp[sample_sub]['task-stim']
    print(f"  Stim runs: {list(sample_stim.keys())[:3]}...")
    first_stim_run = list(sample_stim.values())[0]
    print(f"  Stim data keys: {list(first_stim_run.keys())}")
    if 'target' in first_stim_run:
        print(f"  Stim target: {first_stim_run['target']}")
    if 'stim time' in first_stim_run:
        print(f"  Stim time type: {type(first_stim_run['stim time'])}")

# Match models to empirical data
subjects_with_models_and_data = []
for sub_id in trained_models.keys():
    if sub_id in dataset_emp:
        subjects_with_models_and_data.append(sub_id)

print(f"\n{'='*70}")
print(f"Subjects with both trained models AND empirical data: {len(subjects_with_models_and_data)}")
print(f"  {subjects_with_models_and_data[:5]}..." if len(subjects_with_models_and_data) > 5 else f"  {subjects_with_models_and_data}")
print(f"{'='*70}")

## Step 5: Simulation Helper Functions

In [ ]:
print("\n" + "="*70)
print("METRIC: Per-Target FC Correlation Analysis (CORTICAL ONLY)")
print("="*70)
print("Using Schaefer 400 cortical ROIs (indices 50-449)...")
print("Computing: Rest FC, Stim FC, Delta FC (stim - rest)\n")

# Extract cortical ROIs only (Schaefer: indices 50-449)
cortical_roi_indices = np.arange(50, 450)
n_cortical = len(cortical_roi_indices)
print(f"Cortical ROIs: {n_cortical} (indices {cortical_roi_indices[0]}-{cortical_roi_indices[-1]})\n")

# Organize FC matrices by target (cortical only)
fc_by_target_emp = {}
fc_by_target_sim = {}

for sub_id in sorted(dataset_sim.keys()):
    if sub_id not in dataset_emp:
        continue

    # Collect rest FC
    ts_rest_emp_list = []
    ts_rest_sim_list = []

    if "task-rest" in dataset_emp[sub_id]:
        for run_idx, run in dataset_emp[sub_id]["task-rest"].items():
            ts = run.get("time series")
            if ts is not None and isinstance(ts, np.ndarray) and ts.shape[1] == ROI_num:
                ts_drop = ts[remove_initial_trs:, :]
                ts_proc = preprocess_run(ts_drop, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)
                if ts_proc.shape[0] > using_steps:
                    ts_cortical = ts_proc[:, cortical_roi_indices]
                    ts_rest_emp_list.append(ts_cortical)

    if "task-rest" in dataset_sim[sub_id]:
        for run_idx, run in dataset_sim[sub_id]["task-rest"].items():
            ts = run.get("time series")
            if ts is not None and isinstance(ts, np.ndarray) and ts.shape[1] == ROI_num:
                ts_proc = preprocess_run(ts, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)
                if ts_proc.shape[0] > using_steps:
                    ts_cortical = ts_proc[:, cortical_roi_indices]
                    ts_rest_sim_list.append(ts_cortical)

    if ts_rest_emp_list and ts_rest_sim_list:
        fc_rest_emp = compute_fc_matrix(np.vstack(ts_rest_emp_list))
        fc_rest_sim = compute_fc_matrix(np.vstack(ts_rest_sim_list))

        # Collect stim FC per target
        if "task-stim" in dataset_sim[sub_id] and "task-stim" in dataset_emp[sub_id]:
            emp_stim = dataset_emp[sub_id]["task-stim"]
            sim_stim = dataset_sim[sub_id]["task-stim"]

            for run_idx in sim_stim.keys():
                if run_idx not in emp_stim:
                    continue

                ts_emp = emp_stim[run_idx].get("time series")
                ts_sim = sim_stim[run_idx].get("time series")
                target_vec = emp_stim[run_idx].get("target")

                if ts_emp is None or ts_sim is None:
                    continue

                target_idx = safe_target_idx(target_vec)
                if target_idx is None:
                    continue

                ts_emp_drop = ts_emp[remove_initial_trs:, :]
                ts_emp_proc = preprocess_run(ts_emp_drop, tr=tr_stim, n_drop=0,
                                            low=low_hz, high=high_hz, order=2, zscore=True)
                ts_sim_proc = preprocess_run(ts_sim, tr=tr_stim, n_drop=0,
                                            low=low_hz, high=high_hz, order=2, zscore=True)

                if ts_emp_proc.shape[0] <= using_steps or ts_sim_proc.shape[0] <= using_steps:
                    continue

                ts_emp_cortical = ts_emp_proc[:, cortical_roi_indices]
                ts_sim_cortical = ts_sim_proc[:, cortical_roi_indices]

                fc_stim_emp = compute_fc_matrix(ts_emp_cortical)
                fc_stim_sim = compute_fc_matrix(ts_sim_cortical)

                if target_idx not in fc_by_target_emp:
                    fc_by_target_emp[target_idx] = {"rest": [], "stim": []}
                    fc_by_target_sim[target_idx] = {"rest": [], "stim": []}

                fc_by_target_emp[target_idx]["rest"].append(fc_rest_emp)
                fc_by_target_emp[target_idx]["stim"].append(fc_stim_emp)
                fc_by_target_sim[target_idx]["rest"].append(fc_rest_sim)
                fc_by_target_sim[target_idx]["stim"].append(fc_stim_sim)

print(f"Processed {len(fc_by_target_emp)} unique stimulation targets\n")

# Compute correlations per target
target_results = {}

for target_idx in sorted(fc_by_target_emp.keys()):
    emp_rest_list = fc_by_target_emp[target_idx]["rest"]
    emp_stim_list = fc_by_target_emp[target_idx]["stim"]
    sim_rest_list = fc_by_target_sim[target_idx]["rest"]
    sim_stim_list = fc_by_target_sim[target_idx]["stim"]

    if len(emp_rest_list) == 0 or len(emp_stim_list) == 0:
        continue

    # Average FC matrices
    mean_fc_rest_emp = np.mean(emp_rest_list, axis=0)
    mean_fc_stim_emp = np.mean(emp_stim_list, axis=0)
    mean_fc_rest_sim = np.mean(sim_rest_list, axis=0)
    mean_fc_stim_sim = np.mean(sim_stim_list, axis=0)

    # Compute delta FC
    mean_fc_delta_emp = mean_fc_stim_emp - mean_fc_rest_emp
    mean_fc_delta_sim = mean_fc_stim_sim - mean_fc_rest_sim

    # Extract upper triangles
    rest_emp_vec = extract_upper_triangle(mean_fc_rest_emp)
    rest_sim_vec = extract_upper_triangle(mean_fc_rest_sim)
    stim_emp_vec = extract_upper_triangle(mean_fc_stim_emp)
    stim_sim_vec = extract_upper_triangle(mean_fc_stim_sim)
    delta_emp_vec = extract_upper_triangle(mean_fc_delta_emp)
    delta_sim_vec = extract_upper_triangle(mean_fc_delta_sim)

    # Compute correlations
    r_rest = np.corrcoef(rest_emp_vec, rest_sim_vec)[0, 1]
    r_stim = np.corrcoef(stim_emp_vec, stim_sim_vec)[0, 1]
    r_delta = np.corrcoef(delta_emp_vec, delta_sim_vec)[0, 1]

    target_results[target_idx] = {
        "r_rest": float(r_rest) if not np.isnan(r_rest) else 0.0,
        "r_stim": float(r_stim) if not np.isnan(r_stim) else 0.0,
        "r_delta": float(r_delta) if not np.isnan(r_delta) else 0.0,
        "n_subjects": len(emp_rest_list),
    }

print(f"Per-Target FC Correlations (CORTICAL ONLY):")
print(f"{'Target':<8} {'r_rest':<12} {'r_stim':<12} {'r_delta':<12} {'N_subj':<8}")
print("-" * 60)

for target_idx in sorted(target_results.keys()):
    res = target_results[target_idx]
    print(f"Target {target_idx:<2} {res['r_rest']:>10.4f}   {res['r_stim']:>10.4f}   {res['r_delta']:>10.4f}   {res['n_subjects']:>6}")

# Summary statistics
all_r_rest = [v["r_rest"] for v in target_results.values()]
all_r_stim = [v["r_stim"] for v in target_results.values()]
all_r_delta = [v["r_delta"] for v in target_results.values()]

print("\n" + "="*70)
print("Overall Summary (NORMAL SIMULATION):")
print(f"  Rest FC:  μ={np.mean(all_r_rest):.4f}, σ={np.std(all_r_rest):.4f}")
print(f"  Stim FC:  μ={np.mean(all_r_stim):.4f}, σ={np.std(all_r_stim):.4f}")
print(f"  Delta FC: μ={np.mean(all_r_delta):.4f}, σ={np.std(all_r_delta):.4f}")
print("="*70)

In [ ]:
# --- Helper Functions for Data Parsing ---

def safe_target_idx(target_vec):
    """Extract target region index from one-hot vector."""
    if target_vec is None:
        return None
    v = np.asarray(target_vec).astype(int).ravel()
    if v.size == 0 or v.sum() != 1:
        return None
    return int(np.argmax(v))

def get_onset_column(df):
    """Find onset column in dataframe."""
    if df is None or len(df) == 0:
        return None
    for col in ["onset", "Onset", "stim_onset", "onset_s", "onset_sec", "time", "t", "seconds"]:
        if col in df.columns:
            return col
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            return col
    return None

def map_onsets_to_steps(onsets_s, tr_model=tr_stim, mode="round"):
    """Map stimulus onsets (seconds) to model steps.

    After removing initial TRs, stimulus times are relative to start of cleaned data.
    This function converts seconds to simulation step indices.
    """
    onsets_s = np.asarray(onsets_s, dtype=float)
    x = onsets_s / float(tr_model)
    if mode == "round":
        steps = np.rint(x).astype(int)
    elif mode == "floor":
        steps = np.floor(x).astype(int)
    else:
        steps = np.ceil(x).astype(int)
    steps = steps[steps >= 0]
    return np.unique(steps)

print("✓ Helper functions defined")

In [ ]:
# --- Neural Network Inference ---

@torch.no_grad()
def predict_next(model, window_SxN):
    """Predict next state from rolling window of brain state.

    For stimulus-agnostic models:
    - Input: flattened brain state from using_steps (1350 dims for 3 steps × 450 ROIs)
    - Output: next predicted ROI activation (450 dims)
    - Noise is added to output for stochasticity

    Args:
        model: ANN_MLP model instance
        window_SxN: (using_steps, ROI_num) array of recent brain states

    Returns:
        (ROI_num,) predicted next state with noise
    """
    # Flatten brain state
    brain_state = window_SxN.reshape(-1).astype(np.float32)

    # Add input noise
    noise = NOISE_SIGMA * rng.normal(0.0, 1.0, size=brain_state.shape).astype(np.float32)
    brain_state = brain_state + noise

    # Forward pass
    x = torch.tensor(brain_state[None, :], dtype=torch.float32, device=device)
    y = model(x)

    return y.detach().cpu().numpy().squeeze(0)

def simulate_run(model, init_window_SxN, n_steps, stim_steps=None, target_idx=None, W=None, stim_amplitude=None):
    """Simulate brain activity time series with optional TMS stimulation.

    The model learns intrinsic dynamics during normal brain function.
    When we apply stimulation, we perturb the state at the target region using:
    - Spatial Gaussian kernel (W) to spread stimulation across regions
    - Empirical stimulus timing (stim_steps) to apply perturbations at correct times

    Args:
        model: Trained ANN_MLP
        init_window_SxN: (using_steps, ROI_num) initial brain state window
        n_steps: Number of simulation steps
        stim_steps: Set of step indices when stimulation occurs
        target_idx: Target ROI for stimulation (0-449)
        W: (ROI_num, ROI_num) spatial Gaussian kernel for TMS spread
        stim_amplitude: Stimulation amplitude. If None, uses global STIM_AMP. Set to 0.0 for control (no perturbation).

    Returns:
        (n_steps, ROI_num) simulated time series
    """
    init_window_SxN = np.asarray(init_window_SxN, dtype=np.float32)
    assert init_window_SxN.shape == (using_steps, ROI_num)

    # Use provided amplitude or default to global STIM_AMP
    if stim_amplitude is None:
        stim_amplitude = STIM_AMP

    # Prepare stimulation parameters
    if stim_steps is not None:
        stim_steps = set(int(s) for s in stim_steps)
    else:
        stim_steps = set()

    do_stim = (target_idx is not None) and (len(stim_steps) > 0)

    w = init_window_SxN.copy()

    # Burn-in phase: let model settle to attractor without stimulation
    for _ in range(BURN_IN):
        y = predict_next(model, w)
        w = np.vstack([w[1:], y[None, :]])

    # Simulate with optional spatial TMS
    out = np.zeros((n_steps, ROI_num), dtype=np.float32)
    for t in range(n_steps):
        # Apply stimulation if this is a stim step
        if do_stim and (t in stim_steps):
            if W is not None:
                # Apply spatial Gaussian spread of stimulation
                w[-1, :] += stim_amplitude * W[target_idx, :]
            else:
                # No spatial kernel - apply to target region only
                w[-1, target_idx] += stim_amplitude

        # Get model prediction
        y = predict_next(model, w)
        out[t] = y

        # Update rolling window
        w = np.vstack([w[1:], y[None, :]])

    return out

print("✓ Simulation functions defined")

## Step 6: Functional Connectivity Computation

In [ ]:
# --- Functional Connectivity Functions ---

def compute_fc_matrix(time_series):
    """Compute functional connectivity (Pearson correlation) matrix.

    Args:
        time_series: (T, ROI_num) time series array

    Returns:
        (ROI_num, ROI_num) correlation matrix
    """
    ts_std = (time_series - time_series.mean(axis=0)) / (time_series.std(axis=0) + 1e-8)
    fc = np.corrcoef(ts_std.T)
    return fc

def extract_upper_triangle(fc_matrix):
    """Extract upper triangle of FC matrix (excluding diagonal) as vector.

    For comparison, this removes redundant lower triangle and diagonal.
    """
    N = fc_matrix.shape[0]
    indices = np.triu_indices(N, k=1)
    return fc_matrix[indices]

print("✓ FC computation functions defined")

## Step 7: Generate Simulated Dataset Using Empirical TMS Parameters

In [ ]:
# --- Generate synthetic dataset ---
print("="*70)
print("GENERATING SIMULATED DATASET")
print("="*70)
print("\nFor each subject with trained model:")
print("  1. Simulate REST sessions (no stimulation)")
print("  2. Simulate STIM sessions (with empirical TMS timing & target)")
print()

dataset_sim = {}
n_sim_rest = 0
n_sim_stim = 0
sim_errors = []

for sub_id in sorted(subjects_with_models_and_data):
    model = trained_models[sub_id]
    sub_data_emp = dataset_emp[sub_id]

    dataset_sim[sub_id] = {"task-rest": {}, "task-stim": {}}

    # ---- SIMULATE REST ----
    if "task-rest" in sub_data_emp:
        rest_count = 0
        for run_idx, run_emp in sub_data_emp["task-rest"].items():
            ts_emp = run_emp.get("time series", None)

            # Validate data
            if ts_emp is None or not isinstance(ts_emp, np.ndarray) or ts_emp.shape[1] != ROI_num:
                continue

            try:
                # Preprocess empirical rest (drop TRs, bandpass, z-score)
                ts_drop = ts_emp[remove_initial_trs:, :]
                ts_proc = preprocess_run(ts_drop, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

                if ts_proc.shape[0] <= using_steps:
                    continue

                # Use first using_steps timesteps as initial window, no stimulation
                init_window = ts_proc[:using_steps].copy()
                n_steps = ts_proc.shape[0]

                # Simulate rest
                sim_ts = simulate_run(model, init_window, n_steps,
                                    stim_steps=None, target_idx=None, W=None)

                dataset_sim[sub_id]["task-rest"][int(run_idx)] = {
                    "time series": sim_ts,
                    "metadata": {"simulated": True, "tr": float(tr_rest), "n_vols": n_steps}
                }
                rest_count += 1
                n_sim_rest += 1
            except Exception as e:
                sim_errors.append(f"{sub_id} rest run {run_idx}: {str(e)[:60]}")

        if rest_count > 0:
            print(f"  {sub_id}: Simulated {rest_count} rest sessions")

    # ---- SIMULATE STIM ----
    if "task-stim" in sub_data_emp:
        stim_count = 0
        for run_idx, run_emp in sub_data_emp["task-stim"].items():
            ts_emp = run_emp.get("time series", None)
            target_vec = run_emp.get("target", None)
            stim_time_df = run_emp.get("stim time", None)

            # Validate data
            if ts_emp is None or not isinstance(ts_emp, np.ndarray) or ts_emp.shape[1] != ROI_num:
                continue

            # Extract target region
            target_idx = safe_target_idx(target_vec)
            if target_idx is None:
                continue

            try:
                # Preprocess empirical stim (drop TRs, bandpass, z-score)
                ts_drop = ts_emp[remove_initial_trs:, :]
                ts_proc = preprocess_run(ts_drop, tr=tr_stim, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

                if ts_proc.shape[0] <= using_steps:
                    continue

                # Extract stimulus timing
                stim_steps_model = []
                if stim_time_df is not None and len(stim_time_df) > 0:
                    onset_col = get_onset_column(stim_time_df)
                    if onset_col is not None:
                        onsets_s = stim_time_df[onset_col].values
                        # Subtract initial TRs that were dropped (in seconds)
                        onsets_s = onsets_s[onsets_s >= remove_initial_trs * tr_stim]
                        onsets_s = onsets_s - remove_initial_trs * tr_stim
                        stim_steps_model = map_onsets_to_steps(onsets_s, tr_model=tr_stim)

                if len(stim_steps_model) == 0:
                    # No valid stimulation times
                    continue

                # Use first using_steps timesteps as initial window
                init_window = ts_proc[:using_steps].copy()
                n_steps = ts_proc.shape[0]

                # Simulate stim with empirical target and timing
                sim_ts = simulate_run(model, init_window, n_steps,
                                    stim_steps=stim_steps_model, target_idx=target_idx, W=W)

                dataset_sim[sub_id]["task-stim"][int(run_idx)] = {
                    "time series": sim_ts,
                    "target": target_vec,
                    "stim time": stim_time_df,
                    "metadata": {
                        "simulated": True,
                        "tr": float(tr_stim),
                        "target_idx": int(target_idx),
                        "n_stim_pulses": len(stim_steps_model),
                        "n_vols": n_steps
                    }
                }
                stim_count += 1
                n_sim_stim += 1
            except Exception as e:
                sim_errors.append(f"{sub_id} stim run {run_idx}: {str(e)[:60]}")

        if stim_count > 0:
            print(f"  {sub_id}: Simulated {stim_count} stim sessions")

print("\n" + "="*70)
print(f"✓ Generated {n_sim_rest} rest sessions and {n_sim_stim} stim sessions")
print(f"✓ Simulated {len(dataset_sim)} subjects total")

if sim_errors:
    print(f"\n⚠ Encountered {len(sim_errors)} errors during simulation:")
    for err in sim_errors[:3]:
        print(f"  {err}")
    if len(sim_errors) > 3:
        print(f"  ... and {len(sim_errors)-3} more")

## Step 7b: Generate Control Dataset - REST INIT for Both Conditions

**KEY INNOVATION**: Both REST and STIM (amplitude=0) use **REST empirical initialization**

This creates a **true amplitude-only comparison**:
- Same initial brain state (from REST)
- Different amplitude (100.0 vs 0.0)

This isolates the effect of stimulation amplitude from initialization effects.

In [ ]:
# --- Generate control dataset with SAME REST INIT for both conditions ---
print("="*70)
print("GENERATING CONTROL DATASET (STIM_AMP = 0, REST INIT FOR BOTH)")
print("="*70)
print("\nFor each subject with trained model:")
print("  1. Simulate REST sessions (no stimulation, from REST init)")
print("  2. Simulate STIM sessions (STIM_AMP=0, FROM SAME REST INIT)")
print("     → Both use REST initialization for true amplitude-only comparison")
print()

STIM_AMP_CONTROL = 0.0  # Control: zero amplitude

dataset_sim_control = {}
n_sim_rest_control = 0
n_sim_stim_control = 0
sim_errors_control = []

for sub_id in sorted(subjects_with_models_and_data):
    model = trained_models[sub_id]
    sub_data_emp = dataset_emp[sub_id]

    dataset_sim_control[sub_id] = {"task-rest": {}, "task-stim": {}}

    # ----- FIRST PASS: COLLECT REST DATA FOR INITIALIZATION -----
    rest_init_windows = []  # Store (init_window, n_steps) pairs from rest sessions
    
    if "task-rest" in sub_data_emp:
        rest_count = 0
        for run_idx, run_emp in sub_data_emp["task-rest"].items():
            ts_emp = run_emp.get("time series", None)

            if ts_emp is None or not isinstance(ts_emp, np.ndarray) or ts_emp.shape[1] != ROI_num:
                continue

            try:
                ts_drop = ts_emp[remove_initial_trs:, :]
                ts_proc = preprocess_run(ts_drop, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

                if ts_proc.shape[0] <= using_steps:
                    continue

                init_window = ts_proc[:using_steps].copy()
                n_steps = ts_proc.shape[0]
                rest_init_windows.append((init_window, n_steps))

                # Simulate REST (from REST init, no stimulation)
                sim_ts = simulate_run(model, init_window, n_steps,
                                    stim_steps=None, target_idx=None, W=None)

                dataset_sim_control[sub_id]["task-rest"][int(run_idx)] = {
                    "time series": sim_ts,
                    "metadata": {"simulated": True, "tr": float(tr_rest), "n_vols": n_steps}
                }
                rest_count += 1
                n_sim_rest_control += 1
            except Exception as e:
                sim_errors_control.append(f"{sub_id} rest run {run_idx}: {str(e)[:60]}")

        if rest_count > 0:
            print(f"  {sub_id}: Simulated {rest_count} rest sessions (from REST init)")

    # ----- SECOND PASS: SIMULATE STIM WITH CONTROL AMPLITUDE (STIM_AMP=0) & REST INIT -----
    if "task-stim" in sub_data_emp and rest_init_windows:
        stim_count = 0
        for run_idx, run_emp in sub_data_emp["task-stim"].items():
            ts_emp = run_emp.get("time series", None)
            target_vec = run_emp.get("target", None)
            stim_time_df = run_emp.get("stim time", None)

            if ts_emp is None or not isinstance(ts_emp, np.ndarray) or ts_emp.shape[1] != ROI_num:
                continue

            target_idx = safe_target_idx(target_vec)
            if target_idx is None:
                continue

            try:
                ts_drop = ts_emp[remove_initial_trs:, :]
                ts_proc = preprocess_run(ts_drop, tr=tr_stim, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

                if ts_proc.shape[0] <= using_steps:
                    continue

                # Extract stimulus timing
                stim_steps_model = []
                if stim_time_df is not None and len(stim_time_df) > 0:
                    onset_col = get_onset_column(stim_time_df)
                    if onset_col is not None:
                        onsets_s = stim_time_df[onset_col].values
                        onsets_s = onsets_s[onsets_s >= remove_initial_trs * tr_stim]
                        onsets_s = onsets_s - remove_initial_trs * tr_stim
                        stim_steps_model = map_onsets_to_steps(onsets_s, tr_model=tr_stim)

                if len(stim_steps_model) == 0:
                    continue

                # USE REST INITIALIZATION (NOT stim initialization)
                # Pick a matching-length rest window for initialization
                n_steps = ts_proc.shape[0]
                matching_rest = [w for w, ns in rest_init_windows if ns >= n_steps]
                
                if not matching_rest:
                    # Use the longest available rest window
                    matching_rest = [max(rest_init_windows, key=lambda x: x[1])[0]]
                
                init_window = matching_rest[0]  # Use first matching REST init

                # CONTROL: Apply STIM_AMP=0 stimulation from REST initialization
                sim_ts = simulate_run(model, init_window, n_steps,
                                    stim_steps=stim_steps_model, target_idx=target_idx, W=W,
                                    stim_amplitude=STIM_AMP_CONTROL)

                dataset_sim_control[sub_id]["task-stim"][int(run_idx)] = {
                    "time series": sim_ts,
                    "target": target_vec,
                    "stim time": stim_time_df,
                    "metadata": {
                        "simulated": True,
                        "tr": float(tr_stim),
                        "target_idx": int(target_idx),
                        "n_stim_pulses": len(stim_steps_model),
                        "n_vols": n_steps,
                        "stim_amp_applied": 0.0,  # Mark as control
                        "init_type": "REST",  # Mark that REST init was used
                    }
                }
                stim_count += 1
                n_sim_stim_control += 1
            except Exception as e:
                sim_errors_control.append(f"{sub_id} stim run {run_idx}: {str(e)[:60]}")

        if stim_count > 0:
            print(f"  {sub_id}: Simulated {stim_count} stim sessions (STIM_AMP=0, REST init)")

print("\n" + "="*70)
print(f"✓ [CONTROL] Generated {n_sim_rest_control} rest sessions and {n_sim_stim_control} stim sessions (STIM_AMP=0)")
print(f"✓ Both REST and STIM (control) use REST empirical initialization")
print(f"✓ Simulated {len(dataset_sim_control)} subjects total")

if sim_errors_control:
    print(f"\n⚠ Encountered {len(sim_errors_control)} errors during control simulation:")
    for err in sim_errors_control[:3]:
        print(f"  {err}")
    if len(sim_errors_control) > 3:
        print(f"  ... and {len(sim_errors_control)-3} more")


## Step 8: Save Simulated Datasets

In [ ]:
# --- Save both datasets ---
print("\nSaving results to preprocessed_subjects_tms_fmri...")

# Save normal dataset
with open(out_pkl, "wb") as f:
    pickle.dump(dataset_sim, f)
print(f"✓ Saved normal simulated dataset to {out_pkl}")

# Save control dataset
out_pkl_control = os.path.join(out_dir, "dataset_simulated_control_REST_INIT_only.pkl")
with open(out_pkl_control, "wb") as f:
    pickle.dump(dataset_sim_control, f)
print(f"✓ Saved control simulated dataset to {out_pkl_control}")

# Create summaries
summary = {
    "metadata": {
        "date": pd.Timestamp.now().isoformat(),
        "model_type": "stimulus-agnostic MLP",
        "n_subjects_simulated": len(dataset_sim),
        "n_rest_sessions": n_sim_rest,
        "n_stim_sessions": n_sim_stim,
    },
    "parameters": {
        "ROI_num": ROI_num,
        "using_steps": using_steps,
        "tr_rest": tr_rest,
        "tr_stim": tr_stim,
        "remove_initial_trs": remove_initial_trs,
        "filter_freq": [low_hz, high_hz],
        "BURN_IN": BURN_IN,
        "NOISE_SIGMA": NOISE_SIGMA,
        "STIM_AMP": STIM_AMP,
        "RHO_MM": RHO_MM,
    },
    "subjects": list(dataset_sim.keys()),
}

with open(results_json, "w") as f:
    json.dump(summary, f, indent=2)
print(f"✓ Saved normal summary to {results_json}")

summary_control = {
    "metadata": {
        "date": pd.Timestamp.now().isoformat(),
        "model_type": "stimulus-agnostic MLP",
        "condition": "CONTROL: STIM_AMP=0, REST INIT for both conditions",
        "n_subjects_simulated": len(dataset_sim_control),
        "n_rest_sessions": n_sim_rest_control,
        "n_stim_sessions": n_sim_stim_control,
    },
    "parameters": {
        "ROI_num": ROI_num,
        "using_steps": using_steps,
        "tr_rest": tr_rest,
        "tr_stim": tr_stim,
        "remove_initial_trs": remove_initial_trs,
        "filter_freq": [low_hz, high_hz],
        "BURN_IN": BURN_IN,
        "NOISE_SIGMA": NOISE_SIGMA,
        "STIM_AMP": STIM_AMP_CONTROL,
        "RHO_MM": RHO_MM,
        "initialization": "REST empirical data for both REST and STIM (control)"
    },
    "subjects": list(dataset_sim_control.keys()),
}

results_json_control = os.path.join(out_dir, "simulation_summary_control_REST_INIT.json")
with open(results_json_control, "w") as f:
    json.dump(summary_control, f, indent=2)
print(f"✓ Saved control summary to {results_json_control}")

print(f"\n{'='*70}")
print(f"NEW SIMULATION PARAMETERS (v2):")
print(f"  BURN_IN: {BURN_IN}")
print(f"  NOISE_SIGMA: {NOISE_SIGMA}")
print(f"  STIM_AMP: {STIM_AMP}")
print(f"  RHO_MM: {RHO_MM}")
print(f"\nKEY DIFFERENCE IN CONTROL:")
print(f"  ✅ Both REST and STIM (amplitude=0) use REST initialization")
print(f"  ✅ True amplitude-only comparison (same brain state, different amplitude)")
print(f"{'='*70}")

## Step 9: Validation - Per-Target FC Analysis (Cortical ROIs Only)

## Step 10: Control Analysis - STIM_AMP=0 with REST Initialization

In [ ]:
print("="*70)
print("CONTROL ANALYSIS: STIM_AMP=0 with REST Initialization")
print("="*70)
print("Computing FC correlations for control (amplitude=0)...\n")

fc_by_target_emp_control = {}
fc_by_target_sim_control = {}

for sub_id in sorted(dataset_sim_control.keys()):
    if sub_id not in dataset_emp:
        continue

    ts_rest_emp_list = []
    ts_rest_sim_list = []

    if "task-rest" in dataset_emp[sub_id]:
        for run_idx, run in dataset_emp[sub_id]["task-rest"].items():
            ts = run.get("time series")
            if ts is not None and isinstance(ts, np.ndarray) and ts.shape[1] == ROI_num:
                ts_drop = ts[remove_initial_trs:, :]
                ts_proc = preprocess_run(ts_drop, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)
                if ts_proc.shape[0] > using_steps:
                    ts_cortical = ts_proc[:, cortical_roi_indices]
                    ts_rest_emp_list.append(ts_cortical)

    if "task-rest" in dataset_sim_control[sub_id]:
        for run_idx, run in dataset_sim_control[sub_id]["task-rest"].items():
            ts = run.get("time series")
            if ts is not None and isinstance(ts, np.ndarray) and ts.shape[1] == ROI_num:
                ts_proc = preprocess_run(ts, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)
                if ts_proc.shape[0] > using_steps:
                    ts_cortical = ts_proc[:, cortical_roi_indices]
                    ts_rest_sim_list.append(ts_cortical)

    if ts_rest_emp_list and ts_rest_sim_list:
        fc_rest_emp = compute_fc_matrix(np.vstack(ts_rest_emp_list))
        fc_rest_sim = compute_fc_matrix(np.vstack(ts_rest_sim_list))

        if "task-stim" in dataset_sim_control[sub_id] and "task-stim" in dataset_emp[sub_id]:
            emp_stim = dataset_emp[sub_id]["task-stim"]
            sim_stim = dataset_sim_control[sub_id]["task-stim"]

            for run_idx in sim_stim.keys():
                if run_idx not in emp_stim:
                    continue

                ts_emp = emp_stim[run_idx].get("time series")
                ts_sim = sim_stim[run_idx].get("time series")
                target_vec = emp_stim[run_idx].get("target")

                if ts_emp is None or ts_sim is None:
                    continue

                target_idx = safe_target_idx(target_vec)
                if target_idx is None:
                    continue

                ts_emp_drop = ts_emp[remove_initial_trs:, :]
                ts_emp_proc = preprocess_run(ts_emp_drop, tr=tr_stim, n_drop=0,
                                            low=low_hz, high=high_hz, order=2, zscore=True)
                ts_sim_proc = preprocess_run(ts_sim, tr=tr_stim, n_drop=0,
                                            low=low_hz, high=high_hz, order=2, zscore=True)

                if ts_emp_proc.shape[0] <= using_steps or ts_sim_proc.shape[0] <= using_steps:
                    continue

                ts_emp_cortical = ts_emp_proc[:, cortical_roi_indices]
                ts_sim_cortical = ts_sim_proc[:, cortical_roi_indices]

                fc_stim_emp = compute_fc_matrix(ts_emp_cortical)
                fc_stim_sim = compute_fc_matrix(ts_sim_cortical)

                if target_idx not in fc_by_target_emp_control:
                    fc_by_target_emp_control[target_idx] = {"rest": [], "stim": []}
                    fc_by_target_sim_control[target_idx] = {"rest": [], "stim": []}

                fc_by_target_emp_control[target_idx]["rest"].append(fc_rest_emp)
                fc_by_target_emp_control[target_idx]["stim"].append(fc_stim_emp)
                fc_by_target_sim_control[target_idx]["rest"].append(fc_rest_sim)
                fc_by_target_sim_control[target_idx]["stim"].append(fc_stim_sim)

print(f"Processed {len(fc_by_target_emp_control)} unique targets (CONTROL)\n")

target_results_control = {}

for target_idx in sorted(fc_by_target_emp_control.keys()):
    emp_rest_list = fc_by_target_emp_control[target_idx]["rest"]
    emp_stim_list = fc_by_target_emp_control[target_idx]["stim"]
    sim_rest_list = fc_by_target_sim_control[target_idx]["rest"]
    sim_stim_list = fc_by_target_sim_control[target_idx]["stim"]

    if len(emp_rest_list) == 0 or len(emp_stim_list) == 0:
        continue

    mean_fc_rest_emp = np.mean(emp_rest_list, axis=0)
    mean_fc_stim_emp = np.mean(emp_stim_list, axis=0)
    mean_fc_rest_sim = np.mean(sim_rest_list, axis=0)
    mean_fc_stim_sim = np.mean(sim_stim_list, axis=0)

    mean_fc_delta_emp = mean_fc_stim_emp - mean_fc_rest_emp
    mean_fc_delta_sim = mean_fc_stim_sim - mean_fc_rest_sim

    rest_emp_vec = extract_upper_triangle(mean_fc_rest_emp)
    rest_sim_vec = extract_upper_triangle(mean_fc_rest_sim)
    stim_emp_vec = extract_upper_triangle(mean_fc_stim_emp)
    stim_sim_vec = extract_upper_triangle(mean_fc_stim_sim)
    delta_emp_vec = extract_upper_triangle(mean_fc_delta_emp)
    delta_sim_vec = extract_upper_triangle(mean_fc_delta_sim)

    r_rest = np.corrcoef(rest_emp_vec, rest_sim_vec)[0, 1]
    r_stim = np.corrcoef(stim_emp_vec, stim_sim_vec)[0, 1]
    r_delta = np.corrcoef(delta_emp_vec, delta_sim_vec)[0, 1]

    target_results_control[target_idx] = {
        "r_rest": float(r_rest) if not np.isnan(r_rest) else 0.0,
        "r_stim": float(r_stim) if not np.isnan(r_stim) else 0.0,
        "r_delta": float(r_delta) if not np.isnan(r_delta) else 0.0,
        "n_subjects": len(emp_rest_list),
    }

all_r_stim_control = [v["r_stim"] for v in target_results_control.values()]
all_r_delta_control = [v["r_delta"] for v in target_results_control.values()]

print(f"Per-Target FC Correlations (CONTROL: STIM_AMP=0):")
print(f"{'Target':<8} {'r_rest':<12} {'r_stim':<12} {'r_delta':<12} {'N_subj':<8}")
print("-" * 60)

for target_idx in sorted(target_results_control.keys()):
    res = target_results_control[target_idx]
    print(f"Target {target_idx:<2} {res['r_rest']:>10.4f}   {res['r_stim']:>10.4f}   {res['r_delta']:>10.4f}   {res['n_subjects']:>6}")

print("\n" + "="*70)
print("COMPARISON SUMMARY")
print("="*70)
print(f"\nNormal (STIM_AMP={STIM_AMP}):")
print(f"  Stim FC:  μ={np.mean(all_r_stim):.4f}")
print(f"  Delta FC: μ={np.mean(all_r_delta):.4f}")

print(f"\nControl (STIM_AMP=0, REST init):")
print(f"  Stim FC:  μ={np.mean(all_r_stim_control):.4f}")
print(f"  Delta FC: μ={np.mean(all_r_delta_control):.4f}")

diff_stim = np.mean(all_r_stim) - np.mean(all_r_stim_control)
diff_delta = np.mean(all_r_delta) - np.mean(all_r_delta_control)

print(f"\nDifference (Normal - Control):")
print(f"  Stim FC:  {diff_stim:+.4f}")
print(f"  Delta FC: {diff_delta:+.4f}")
print("="*70)

## Step 11: Comparative Visualization - STIM_AMP vs STIM_AMP=0

## Step 8: Session-Specific FC Correlation (CORTICAL ONLY)

In [ ]:
print("\n" + "="*70)
print("METRIC 1: Session-Specific FC Correlation (CORTICAL AREAS ONLY)")
print("="*70)
print("Using last 400 Schaefer ROIs (cortical), excluding first 50 Tian ROIs (subcortical)")

# Extract cortical ROIs only (Schaefer: indices 50-449)
cortical_roi_indices = np.arange(50, 450)
n_cortical = len(cortical_roi_indices)
print(f"Cortical ROIs: {n_cortical} (indices {cortical_roi_indices[0]}-{cortical_roi_indices[-1]})\n")

# Session-specific FC correlations
fc_corr_rest = {}  # sub_id -> [FC correlations per rest session]
fc_corr_stim = {}  # sub_id -> [FC correlations per stim session]

for sub_id in sorted(dataset_sim.keys()):
    if sub_id not in dataset_emp:
        continue

    fc_corr_rest[sub_id] = []
    fc_corr_stim[sub_id] = []

    # ---- REST FC ----
    if "task-rest" in dataset_sim[sub_id] and "task-rest" in dataset_emp[sub_id]:
        emp_rest = dataset_emp[sub_id]["task-rest"]
        sim_rest = dataset_sim[sub_id]["task-rest"]

        for run_idx in sim_rest.keys():
            if run_idx not in emp_rest:
                continue

            ts_emp = emp_rest[run_idx].get("time series")
            ts_sim = sim_rest[run_idx].get("time series")

            if ts_emp is None or ts_sim is None:
                continue

            # Preprocess
            ts_emp_drop = ts_emp[remove_initial_trs:, :]
            ts_emp_proc = preprocess_run(ts_emp_drop, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)
            ts_sim_proc = preprocess_run(ts_sim, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

            # Ensure same length
            min_len = min(ts_emp_proc.shape[0], ts_sim_proc.shape[0])
            ts_emp_proc = ts_emp_proc[:min_len]
            ts_sim_proc = ts_sim_proc[:min_len]

            # Extract cortical ROIs only
            ts_emp_cortical = ts_emp_proc[:, cortical_roi_indices]
            ts_sim_cortical = ts_sim_proc[:, cortical_roi_indices]

            # Compute FC matrices
            fc_emp = compute_fc_matrix(ts_emp_cortical)
            fc_sim = compute_fc_matrix(ts_sim_cortical)

            # Extract upper triangle and correlate
            fc_emp_vec = extract_upper_triangle(fc_emp)
            fc_sim_vec = extract_upper_triangle(fc_sim)

            r_fc = np.corrcoef(fc_emp_vec, fc_sim_vec)[0, 1]
            if not np.isnan(r_fc):
                fc_corr_rest[sub_id].append(r_fc)

    # ---- STIM FC ----
    if "task-stim" in dataset_sim[sub_id] and "task-stim" in dataset_emp[sub_id]:
        emp_stim = dataset_emp[sub_id]["task-stim"]
        sim_stim = dataset_sim[sub_id]["task-stim"]

        for run_idx in sim_stim.keys():
            if run_idx not in emp_stim:
                continue

            ts_emp = emp_stim[run_idx].get("time series")
            ts_sim = sim_stim[run_idx].get("time series")

            if ts_emp is None or ts_sim is None:
                continue

            # Preprocess
            ts_emp_drop = ts_emp[remove_initial_trs:, :]
            ts_emp_proc = preprocess_run(ts_emp_drop, tr=tr_stim, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)
            ts_sim_proc = preprocess_run(ts_sim, tr=tr_stim, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

            # Ensure same length
            min_len = min(ts_emp_proc.shape[0], ts_sim_proc.shape[0])
            ts_emp_proc = ts_emp_proc[:min_len]
            ts_sim_proc = ts_sim_proc[:min_len]

            # Extract cortical ROIs only
            ts_emp_cortical = ts_emp_proc[:, cortical_roi_indices]
            ts_sim_cortical = ts_sim_proc[:, cortical_roi_indices]

            # Compute FC matrices
            fc_emp = compute_fc_matrix(ts_emp_cortical)
            fc_sim = compute_fc_matrix(ts_sim_cortical)

            # Extract upper triangle and correlate
            fc_emp_vec = extract_upper_triangle(fc_emp)
            fc_sim_vec = extract_upper_triangle(fc_sim)

            r_fc = np.corrcoef(fc_emp_vec, fc_sim_vec)[0, 1]
            if not np.isnan(r_fc):
                fc_corr_stim[sub_id].append(r_fc)

# Summary
print("\nSession-specific FC correlations (CORTICAL ONLY):")
print("\nREST:")
all_rest_fc_corrs = []
for sub_id, corrs in fc_corr_rest.items():
    if corrs:
        mean_corr = np.mean(corrs)
        all_rest_fc_corrs.extend(corrs)
        print(f"  {sub_id}: {mean_corr:.4f} ± {np.std(corrs):.4f} (n={len(corrs)} sessions)")

if all_rest_fc_corrs:
    print(f"\nREST Overall: {np.mean(all_rest_fc_corrs):.4f} ± {np.std(all_rest_fc_corrs):.4f} (n={len(all_rest_fc_corrs)} sessions)")

print("\nSTIM:")
all_stim_fc_corrs = []
for sub_id, corrs in fc_corr_stim.items():
    if corrs:
        mean_corr = np.mean(corrs)
        all_stim_fc_corrs.extend(corrs)
        print(f"  {sub_id}: {mean_corr:.4f} ± {np.std(corrs):.4f} (n={len(corrs)} sessions)")

if all_stim_fc_corrs:
    print(f"\nSTIM Overall: {np.mean(all_stim_fc_corrs):.4f} ± {np.std(all_stim_fc_corrs):.4f} (n={len(all_stim_fc_corrs)} sessions)")

In [ ]:
print("\n" + "="*70)
print("CONTROL DATASET: Session-Specific FC Correlation (CORTICAL ONLY)")
print("="*70)
print("(STIM_AMP=0 with REST initialization)")

# Session-specific FC correlations for CONTROL dataset
fc_corr_rest_ctrl = {}  # sub_id -> [FC correlations per control rest session]
fc_corr_stim_ctrl = {}  # sub_id -> [FC correlations per control stim session]

for sub_id in sorted(dataset_sim_ctrl.keys()):
    if sub_id not in dataset_emp:
        continue

    fc_corr_rest_ctrl[sub_id] = []
    fc_corr_stim_ctrl[sub_id] = []

    # ---- REST FC (CONTROL) ----
    if "task-rest" in dataset_sim_ctrl[sub_id] and "task-rest" in dataset_emp[sub_id]:
        emp_rest = dataset_emp[sub_id]["task-rest"]
        sim_rest = dataset_sim_ctrl[sub_id]["task-rest"]

        for run_idx in sim_rest.keys():
            if run_idx not in emp_rest:
                continue

            ts_emp = emp_rest[run_idx].get("time series")
            ts_sim = sim_rest[run_idx].get("time series")

            if ts_emp is None or ts_sim is None:
                continue

            # Preprocess
            ts_emp_drop = ts_emp[remove_initial_trs:, :]
            ts_emp_proc = preprocess_run(ts_emp_drop, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)
            ts_sim_proc = preprocess_run(ts_sim, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

            # Ensure same length
            min_len = min(ts_emp_proc.shape[0], ts_sim_proc.shape[0])
            ts_emp_proc = ts_emp_proc[:min_len]
            ts_sim_proc = ts_sim_proc[:min_len]

            # Extract cortical ROIs only
            ts_emp_cortical = ts_emp_proc[:, cortical_roi_indices]
            ts_sim_cortical = ts_sim_proc[:, cortical_roi_indices]

            # Compute FC matrices
            fc_emp = compute_fc_matrix(ts_emp_cortical)
            fc_sim = compute_fc_matrix(ts_sim_cortical)

            # Extract upper triangle and correlate
            fc_emp_vec = extract_upper_triangle(fc_emp)
            fc_sim_vec = extract_upper_triangle(fc_sim)

            r_fc = np.corrcoef(fc_emp_vec, fc_sim_vec)[0, 1]
            if not np.isnan(r_fc):
                fc_corr_rest_ctrl[sub_id].append(r_fc)

    # ---- STIM FC (CONTROL) ----
    if "task-stim" in dataset_sim_ctrl[sub_id] and "task-stim" in dataset_emp[sub_id]:
        emp_stim = dataset_emp[sub_id]["task-stim"]
        sim_stim = dataset_sim_ctrl[sub_id]["task-stim"]

        for run_idx in sim_stim.keys():
            if run_idx not in emp_stim:
                continue

            ts_emp = emp_stim[run_idx].get("time series")
            ts_sim = sim_stim[run_idx].get("time series")

            if ts_emp is None or ts_sim is None:
                continue

            # Preprocess
            ts_emp_drop = ts_emp[remove_initial_trs:, :]
            ts_emp_proc = preprocess_run(ts_emp_drop, tr=tr_stim, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)
            ts_sim_proc = preprocess_run(ts_sim, tr=tr_stim, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

            # Ensure same length
            min_len = min(ts_emp_proc.shape[0], ts_sim_proc.shape[0])
            ts_emp_proc = ts_emp_proc[:min_len]
            ts_sim_proc = ts_sim_proc[:min_len]

            # Extract cortical ROIs only
            ts_emp_cortical = ts_emp_proc[:, cortical_roi_indices]
            ts_sim_cortical = ts_sim_proc[:, cortical_roi_indices]

            # Compute FC matrices
            fc_emp = compute_fc_matrix(ts_emp_cortical)
            fc_sim = compute_fc_matrix(ts_sim_cortical)

            # Extract upper triangle and correlate
            fc_emp_vec = extract_upper_triangle(fc_emp)
            fc_sim_vec = extract_upper_triangle(fc_sim)

            r_fc = np.corrcoef(fc_emp_vec, fc_sim_vec)[0, 1]
            if not np.isnan(r_fc):
                fc_corr_stim_ctrl[sub_id].append(r_fc)

# Summary
print("\nSession-specific FC correlations (CORTICAL ONLY) - CONTROL:")
print("\nREST (CONTROL):")
all_rest_fc_corrs_ctrl = []
for sub_id, corrs in fc_corr_rest_ctrl.items():
    if corrs:
        mean_corr = np.mean(corrs)
        all_rest_fc_corrs_ctrl.extend(corrs)
        print(f"  {sub_id}: {mean_corr:.4f} ± {np.std(corrs):.4f} (n={len(corrs)} sessions)")

if all_rest_fc_corrs_ctrl:
    print(f"\nREST (CONTROL) Overall: {np.mean(all_rest_fc_corrs_ctrl):.4f} ± {np.std(all_rest_fc_corrs_ctrl):.4f} (n={len(all_rest_fc_corrs_ctrl)} sessions)")

print("\nSTIM (CONTROL):")
all_stim_fc_corrs_ctrl = []
for sub_id, corrs in fc_corr_stim_ctrl.items():
    if corrs:
        mean_corr = np.mean(corrs)
        all_stim_fc_corrs_ctrl.extend(corrs)
        print(f"  {sub_id}: {mean_corr:.4f} ± {np.std(corrs):.4f} (n={len(corrs)} sessions)")

if all_stim_fc_corrs_ctrl:
    print(f"\nSTIM (CONTROL) Overall: {np.mean(all_stim_fc_corrs_ctrl):.4f} ± {np.std(all_stim_fc_corrs_ctrl):.4f} (n={len(all_stim_fc_corrs_ctrl)} sessions)")

## Step 9: Per-Target FC Analysis

In [ ]:
print("\n" + "="*70)
print("Per-Target FC Analysis")
print("="*70)

# Determine targets from onsets
targets_all = set()
for sub_id in dataset_sim.keys():
    if "task-stim" in dataset_sim[sub_id]:
        for run_idx in dataset_sim[sub_id]["task-stim"].keys():
            metadata = dataset_sim[sub_id]["task-stim"][run_idx].get("metadata", {})
            targets_all.update(metadata.get("targets", []))

targets_list = sorted(list(targets_all))
print(f"Found {len(targets_list)} targets: {targets_list}\n")

# Per-target analysis
target_rest_fc = {target: [] for target in targets_list}  # List of (sub, corr) tuples
target_stim_fc = {target: [] for target in targets_list}
target_delta_fc = {target: [] for target in targets_list}

for sub_id in sorted(dataset_sim.keys()):
    if sub_id not in dataset_emp:
        continue

    # ---- REST FC by target ----
    if "task-rest" in dataset_sim[sub_id] and "task-rest" in dataset_emp[sub_id]:
        emp_rest = dataset_emp[sub_id]["task-rest"]
        sim_rest = dataset_sim[sub_id]["task-rest"]

        rest_fc_corrs = []
        for run_idx in sim_rest.keys():
            if run_idx not in emp_rest:
                continue

            ts_emp = emp_rest[run_idx].get("time series")
            ts_sim = sim_rest[run_idx].get("time series")
            if ts_emp is None or ts_sim is None:
                continue

            ts_emp_drop = ts_emp[remove_initial_trs:, :]
            ts_emp_proc = preprocess_run(ts_emp_drop, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)
            ts_sim_proc = preprocess_run(ts_sim, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

            min_len = min(ts_emp_proc.shape[0], ts_sim_proc.shape[0])
            ts_emp_proc = ts_emp_proc[:min_len]
            ts_sim_proc = ts_sim_proc[:min_len]

            fc_emp = compute_fc_matrix(ts_emp_proc)
            fc_sim = compute_fc_matrix(ts_sim_proc)
            fc_emp_vec = extract_upper_triangle(fc_emp)
            fc_sim_vec = extract_upper_triangle(fc_sim)
            r_fc = np.corrcoef(fc_emp_vec, fc_sim_vec)[0, 1]
            if not np.isnan(r_fc):
                rest_fc_corrs.append(r_fc)

        # Store average across rest runs for all targets
        if rest_fc_corrs:
            avg_rest_fc = np.mean(rest_fc_corrs)
            for target in targets_list:
                target_rest_fc[target].append((sub_id, avg_rest_fc))

    # ---- STIM FC by target ----
    if "task-stim" in dataset_sim[sub_id] and "task-stim" in dataset_emp[sub_id]:
        emp_stim = dataset_emp[sub_id]["task-stim"]
        sim_stim = dataset_sim[sub_id]["task-stim"]

        for target in targets_list:
            stim_fc_corrs = []

            for run_idx in sim_stim.keys():
                if run_idx not in emp_stim:
                    continue

                metadata = sim_stim[run_idx].get("metadata", {})
                if target not in metadata.get("targets", []):
                    continue

                ts_emp = emp_stim[run_idx].get("time series")
                ts_sim = sim_stim[run_idx].get("time series")
                if ts_emp is None or ts_sim is None:
                    continue

                ts_emp_drop = ts_emp[remove_initial_trs:, :]
                ts_emp_proc = preprocess_run(ts_emp_drop, tr=tr_stim, n_drop=0,
                                            low=low_hz, high=high_hz, order=2, zscore=True)
                ts_sim_proc = preprocess_run(ts_sim, tr=tr_stim, n_drop=0,
                                            low=low_hz, high=high_hz, order=2, zscore=True)

                min_len = min(ts_emp_proc.shape[0], ts_sim_proc.shape[0])
                ts_emp_proc = ts_emp_proc[:min_len]
                ts_sim_proc = ts_sim_proc[:min_len]

                fc_emp = compute_fc_matrix(ts_emp_proc)
                fc_sim = compute_fc_matrix(ts_sim_proc)
                fc_emp_vec = extract_upper_triangle(fc_emp)
                fc_sim_vec = extract_upper_triangle(fc_sim)
                r_fc = np.corrcoef(fc_emp_vec, fc_sim_vec)[0, 1]
                if not np.isnan(r_fc):
                    stim_fc_corrs.append(r_fc)

            if stim_fc_corrs:
                avg_stim_fc = np.mean(stim_fc_corrs)
                # Find the corresponding rest FC for delta computation
                for sub_id_match, rest_fc in target_rest_fc[target]:
                    if sub_id_match == sub_id:
                        target_stim_fc[target].append((sub_id, avg_stim_fc))
                        target_delta_fc[target].append((sub_id, avg_stim_fc - rest_fc))
                        break

# Print summary
print("Per-Target FC Metrics:")
for target in targets_list:
    rest_vals = [v for _, v in target_rest_fc[target]]
    stim_vals = [v for _, v in target_stim_fc[target]]
    delta_vals = [v for _, v in target_delta_fc[target]]

    print(f"\nTarget {target}:")
    if rest_vals:
        print(f"  Rest FC: {np.mean(rest_vals):.4f} ± {np.std(rest_vals):.4f}")
    if stim_vals:
        print(f"  Stim FC: {np.mean(stim_vals):.4f} ± {np.std(stim_vals):.4f}")
    if delta_vals:
        print(f"  Delta FC: {np.mean(delta_vals):.4f} ± {np.std(delta_vals):.4f}")

### Per-Target Visualization (Normal Dataset)

In [ ]:
import matplotlib.gridspec as gridspec

# Create per-target figures for normal dataset
per_target_results = {}

for target in targets_list:
    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)

    rest_vals = np.array([v for _, v in target_rest_fc[target]])
    stim_vals = np.array([v for _, v in target_stim_fc[target]])
    delta_vals = np.array([v for _, v in target_delta_fc[target]])

    # Row 1: Histograms
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.hist(rest_vals, bins=10, alpha=0.7, color='blue', edgecolor='black')
    ax1.set_xlabel('FC Correlation')
    ax1.set_ylabel('Frequency')
    ax1.set_title(f'Target {target}: Rest FC\n(n={len(rest_vals)})')
    ax1.axvline(np.mean(rest_vals), color='darkblue', linestyle='--', linewidth=2, label=f'Mean={np.mean(rest_vals):.3f}')
    ax1.legend()

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(stim_vals, bins=10, alpha=0.7, color='red', edgecolor='black')
    ax2.set_xlabel('FC Correlation')
    ax2.set_ylabel('Frequency')
    ax2.set_title(f'Target {target}: Stim FC\n(n={len(stim_vals)})')
    ax2.axvline(np.mean(stim_vals), color='darkred', linestyle='--', linewidth=2, label=f'Mean={np.mean(stim_vals):.3f}')
    ax2.legend()

    ax3 = fig.add_subplot(gs[0, 2])
    ax3.hist(delta_vals, bins=10, alpha=0.7, color='green', edgecolor='black')
    ax3.set_xlabel('Delta FC Correlation')
    ax3.set_ylabel('Frequency')
    ax3.set_title(f'Target {target}: Delta FC (Stim-Rest)\n(n={len(delta_vals)})')
    ax3.axvline(np.mean(delta_vals), color='darkgreen', linestyle='--', linewidth=2, label=f'Mean={np.mean(delta_vals):.3f}')
    ax3.axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.3)
    ax3.legend()

    # Row 2: Scatter plots and summary
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.scatter(rest_vals, stim_vals, alpha=0.6, s=50)
    ax4.set_xlabel('Rest FC')
    ax4.set_ylabel('Stim FC')
    ax4.set_title(f'Target {target}: Rest vs Stim FC')
    min_val = min(rest_vals.min(), stim_vals.min())
    max_val = max(rest_vals.max(), stim_vals.max())
    ax4.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.3)
    ax4.grid(True, alpha=0.3)

    # Bar chart comparing Rest, Stim, and Delta
    ax5 = fig.add_subplot(gs[1, 1])
    categories = ['Rest FC', 'Stim FC', 'Delta FC']
    means = [np.mean(rest_vals), np.mean(stim_vals), np.mean(delta_vals)]
    stds = [np.std(rest_vals), np.std(stim_vals), np.std(delta_vals)]
    colors = ['blue', 'red', 'green']
    bars = ax5.bar(categories, means, yerr=stds, capsize=5, color=colors, alpha=0.7, edgecolor='black')
    ax5.set_ylabel('FC Correlation')
    ax5.set_title(f'Target {target}: Mean ± Std')
    ax5.grid(True, axis='y', alpha=0.3)
    for i, (mean, std) in enumerate(zip(means, stds)):
        ax5.text(i, mean + std + 0.01, f'{mean:.3f}', ha='center', va='bottom', fontsize=9)

    # Summary text box
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.axis('off')
    summary_text = f"""
    Target {target} Summary (Normal Dataset)
    
    REST FC:
      Mean: {np.mean(rest_vals):.4f}
      Std:  {np.std(rest_vals):.4f}
      N:    {len(rest_vals)}
    
    STIM FC:
      Mean: {np.mean(stim_vals):.4f}
      Std:  {np.std(stim_vals):.4f}
      N:    {len(stim_vals)}
    
    DELTA FC:
      Mean: {np.mean(delta_vals):.4f}
      Std:  {np.std(delta_vals):.4f}
      N:    {len(delta_vals)}
    """
    ax6.text(0.1, 0.9, summary_text, transform=ax6.transAxes, fontsize=10,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle(f'Per-Target FC Analysis: Target {target} (Normal Dataset)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'target_{target}_fc_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    per_target_results[target] = {
        'rest_mean': np.mean(rest_vals),
        'rest_std': np.std(rest_vals),
        'stim_mean': np.mean(stim_vals),
        'stim_std': np.std(stim_vals),
        'delta_mean': np.mean(delta_vals),
        'delta_std': np.std(delta_vals),
        'n_subjects': len(rest_vals)
    }

print("\nPer-target visualization complete.")

## Step 10: Control Dataset Per-Target Analysis

In [ ]:
print("\n" + "="*70)
print("Per-Target FC Analysis (CONTROL Dataset)")
print("="*70)

# Per-target analysis for CONTROL dataset
target_rest_fc_ctrl = {target: [] for target in targets_list}
target_stim_fc_ctrl = {target: [] for target in targets_list}
target_delta_fc_ctrl = {target: [] for target in targets_list}

for sub_id in sorted(dataset_sim_ctrl.keys()):
    if sub_id not in dataset_emp:
        continue

    # ---- REST FC by target (CONTROL) ----
    if "task-rest" in dataset_sim_ctrl[sub_id] and "task-rest" in dataset_emp[sub_id]:
        emp_rest = dataset_emp[sub_id]["task-rest"]
        sim_rest = dataset_sim_ctrl[sub_id]["task-rest"]

        rest_fc_corrs = []
        for run_idx in sim_rest.keys():
            if run_idx not in emp_rest:
                continue

            ts_emp = emp_rest[run_idx].get("time series")
            ts_sim = sim_rest[run_idx].get("time series")
            if ts_emp is None or ts_sim is None:
                continue

            ts_emp_drop = ts_emp[remove_initial_trs:, :]
            ts_emp_proc = preprocess_run(ts_emp_drop, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)
            ts_sim_proc = preprocess_run(ts_sim, tr=tr_rest, n_drop=0,
                                        low=low_hz, high=high_hz, order=2, zscore=True)

            min_len = min(ts_emp_proc.shape[0], ts_sim_proc.shape[0])
            ts_emp_proc = ts_emp_proc[:min_len]
            ts_sim_proc = ts_sim_proc[:min_len]

            fc_emp = compute_fc_matrix(ts_emp_proc)
            fc_sim = compute_fc_matrix(ts_sim_proc)
            fc_emp_vec = extract_upper_triangle(fc_emp)
            fc_sim_vec = extract_upper_triangle(fc_sim)
            r_fc = np.corrcoef(fc_emp_vec, fc_sim_vec)[0, 1]
            if not np.isnan(r_fc):
                rest_fc_corrs.append(r_fc)

        if rest_fc_corrs:
            avg_rest_fc = np.mean(rest_fc_corrs)
            for target in targets_list:
                target_rest_fc_ctrl[target].append((sub_id, avg_rest_fc))

    # ---- STIM FC by target (CONTROL) ----
    if "task-stim" in dataset_sim_ctrl[sub_id] and "task-stim" in dataset_emp[sub_id]:
        emp_stim = dataset_emp[sub_id]["task-stim"]
        sim_stim = dataset_sim_ctrl[sub_id]["task-stim"]

        for target in targets_list:
            stim_fc_corrs = []

            for run_idx in sim_stim.keys():
                if run_idx not in emp_stim:
                    continue

                metadata = sim_stim[run_idx].get("metadata", {})
                if target not in metadata.get("targets", []):
                    continue

                ts_emp = emp_stim[run_idx].get("time series")
                ts_sim = sim_stim[run_idx].get("time series")
                if ts_emp is None or ts_sim is None:
                    continue

                ts_emp_drop = ts_emp[remove_initial_trs:, :]
                ts_emp_proc = preprocess_run(ts_emp_drop, tr=tr_stim, n_drop=0,
                                            low=low_hz, high=high_hz, order=2, zscore=True)
                ts_sim_proc = preprocess_run(ts_sim, tr=tr_stim, n_drop=0,
                                            low=low_hz, high=high_hz, order=2, zscore=True)

                min_len = min(ts_emp_proc.shape[0], ts_sim_proc.shape[0])
                ts_emp_proc = ts_emp_proc[:min_len]
                ts_sim_proc = ts_sim_proc[:min_len]

                fc_emp = compute_fc_matrix(ts_emp_proc)
                fc_sim = compute_fc_matrix(ts_sim_proc)
                fc_emp_vec = extract_upper_triangle(fc_emp)
                fc_sim_vec = extract_upper_triangle(fc_sim)
                r_fc = np.corrcoef(fc_emp_vec, fc_sim_vec)[0, 1]
                if not np.isnan(r_fc):
                    stim_fc_corrs.append(r_fc)

            if stim_fc_corrs:
                avg_stim_fc = np.mean(stim_fc_corrs)
                for sub_id_match, rest_fc in target_rest_fc_ctrl[target]:
                    if sub_id_match == sub_id:
                        target_stim_fc_ctrl[target].append((sub_id, avg_stim_fc))
                        target_delta_fc_ctrl[target].append((sub_id, avg_stim_fc - rest_fc))
                        break

# Print summary
print("Per-Target FC Metrics (CONTROL):")
per_target_results_ctrl = {}
for target in targets_list:
    rest_vals = np.array([v for _, v in target_rest_fc_ctrl[target]])
    stim_vals = np.array([v for _, v in target_stim_fc_ctrl[target]])
    delta_vals = np.array([v for _, v in target_delta_fc_ctrl[target]])

    print(f"\nTarget {target}:")
    if rest_vals.size > 0:
        print(f"  Rest FC: {np.mean(rest_vals):.4f} ± {np.std(rest_vals):.4f}")
    if stim_vals.size > 0:
        print(f"  Stim FC: {np.mean(stim_vals):.4f} ± {np.std(stim_vals):.4f}")
    if delta_vals.size > 0:
        print(f"  Delta FC: {np.mean(delta_vals):.4f} ± {np.std(delta_vals):.4f}")

    per_target_results_ctrl[target] = {
        'rest_mean': np.mean(rest_vals) if rest_vals.size > 0 else np.nan,
        'rest_std': np.std(rest_vals) if rest_vals.size > 0 else np.nan,
        'stim_mean': np.mean(stim_vals) if stim_vals.size > 0 else np.nan,
        'stim_std': np.std(stim_vals) if stim_vals.size > 0 else np.nan,
        'delta_mean': np.mean(delta_vals) if delta_vals.size > 0 else np.nan,
        'delta_std': np.std(delta_vals) if delta_vals.size > 0 else np.nan,
        'n_subjects': len(rest_vals)
    }

### Per-Target Visualization (Control Dataset - STIM_AMP=0)

In [ ]:
# Visualize per-target FC for control dataset
for target in targets_list:
    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)

    rest_vals = np.array([v for _, v in target_rest_fc_ctrl[target]])
    stim_vals = np.array([v for _, v in target_stim_fc_ctrl[target]])
    delta_vals = np.array([v for _, v in target_delta_fc_ctrl[target]])

    # Row 1: Histograms
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.hist(rest_vals, bins=10, alpha=0.7, color='blue', edgecolor='black')
    ax1.set_xlabel('FC Correlation')
    ax1.set_ylabel('Frequency')
    ax1.set_title(f'Target {target}: Rest FC (CTRL)\n(n={len(rest_vals)})')
    if len(rest_vals) > 0:
        ax1.axvline(np.mean(rest_vals), color='darkblue', linestyle='--', linewidth=2, label=f'Mean={np.mean(rest_vals):.3f}')
        ax1.legend()

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(stim_vals, bins=10, alpha=0.7, color='red', edgecolor='black')
    ax2.set_xlabel('FC Correlation')
    ax2.set_ylabel('Frequency')
    ax2.set_title(f'Target {target}: Stim FC (CTRL)\n(n={len(stim_vals)})')
    if len(stim_vals) > 0:
        ax2.axvline(np.mean(stim_vals), color='darkred', linestyle='--', linewidth=2, label=f'Mean={np.mean(stim_vals):.3f}')
        ax2.legend()

    ax3 = fig.add_subplot(gs[0, 2])
    ax3.hist(delta_vals, bins=10, alpha=0.7, color='green', edgecolor='black')
    ax3.set_xlabel('Delta FC Correlation')
    ax3.set_ylabel('Frequency')
    ax3.set_title(f'Target {target}: Delta FC (CTRL)\n(n={len(delta_vals)})')
    if len(delta_vals) > 0:
        ax3.axvline(np.mean(delta_vals), color='darkgreen', linestyle='--', linewidth=2, label=f'Mean={np.mean(delta_vals):.3f}')
        ax3.axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.3)
        ax3.legend()

    # Row 2: Scatter and summary
    ax4 = fig.add_subplot(gs[1, 0])
    if len(rest_vals) > 0 and len(stim_vals) > 0:
        ax4.scatter(rest_vals, stim_vals, alpha=0.6, s=50)
        ax4.set_xlabel('Rest FC')
        ax4.set_ylabel('Stim FC')
        ax4.set_title(f'Target {target}: Rest vs Stim FC (CTRL)')
        min_val = min(rest_vals.min(), stim_vals.min())
        max_val = max(rest_vals.max(), stim_vals.max())
        ax4.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.3)
        ax4.grid(True, alpha=0.3)

    # Bar chart
    ax5 = fig.add_subplot(gs[1, 1])
    categories = ['Rest FC', 'Stim FC', 'Delta FC']
    means = [np.mean(rest_vals) if len(rest_vals) > 0 else 0,
             np.mean(stim_vals) if len(stim_vals) > 0 else 0,
             np.mean(delta_vals) if len(delta_vals) > 0 else 0]
    stds = [np.std(rest_vals) if len(rest_vals) > 0 else 0,
            np.std(stim_vals) if len(stim_vals) > 0 else 0,
            np.std(delta_vals) if len(delta_vals) > 0 else 0]
    colors = ['blue', 'red', 'green']
    bars = ax5.bar(categories, means, yerr=stds, capsize=5, color=colors, alpha=0.7, edgecolor='black')
    ax5.set_ylabel('FC Correlation')
    ax5.set_title(f'Target {target}: Mean ± Std (CTRL)')
    ax5.grid(True, axis='y', alpha=0.3)
    for i, (mean, std) in enumerate(zip(means, stds)):
        if mean != 0:
            ax5.text(i, mean + std + 0.01, f'{mean:.3f}', ha='center', va='bottom', fontsize=9)

    # Summary text box
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.axis('off')
    summary_text = f"""
    Target {target} Summary (Control Dataset)
    
    REST FC:
      Mean: {np.mean(rest_vals):.4f}
      Std:  {np.std(rest_vals):.4f}
      N:    {len(rest_vals)}
    
    STIM FC:
      Mean: {np.mean(stim_vals):.4f}
      Std:  {np.std(stim_vals):.4f}
      N:    {len(stim_vals)}
    
    DELTA FC:
      Mean: {np.mean(delta_vals):.4f}
      Std:  {np.std(delta_vals):.4f}
      N:    {len(delta_vals)}
    """
    ax6.text(0.1, 0.9, summary_text, transform=ax6.transAxes, fontsize=10,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

    plt.suptitle(f'Per-Target FC Analysis: Target {target} (Control Dataset - STIM_AMP=0)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'target_{target}_fc_analysis_control.png', dpi=150, bbox_inches='tight')
    plt.show()

print("Control dataset per-target visualization complete.")

## Step 11: Normal vs Control Comparison

In [ ]:
print("\n" + "="*70)
print("COMPARISON: Normal vs Control Dataset")
print("="*70)
print("\nNormal: STIM_AMP=10.0 (Full amplitude effect)")
print("Control: STIM_AMP=0 with REST initialization (Baseline, no amplitude effect)")
print("\nQuestion: Where does the amplitude effect (STIM_AMP=10 vs STIM_AMP=0) manifest?")
print("Expected: Larger Delta FC in targets with strong amplitude modulation\n")

# Comparison table
comparison_data = []
for target in targets_list:
    normal = per_target_results[target]
    control = per_target_results_ctrl[target]

    delta_diff = normal['delta_mean'] - control['delta_mean']

    comparison_data.append({
        'target': target,
        'normal_delta_mean': normal['delta_mean'],
        'control_delta_mean': control['delta_mean'],
        'delta_difference': delta_diff,
        'pct_change': (delta_diff / abs(control['delta_mean']) * 100) if control['delta_mean'] != 0 else 0,
        'normal_n': normal['n_subjects'],
        'control_n': control['n_subjects']
    })

# Sort by delta_difference
comparison_data.sort(key=lambda x: x['delta_difference'], reverse=True)

print("Target | Normal Delta FC | Control Delta FC | Difference | % Change")
print("-" * 75)
for row in comparison_data:
    print(f"{row['target']:6} | {row['normal_delta_mean']:15.4f} | {row['control_delta_mean']:16.4f} | {row['delta_difference']:10.4f} | {row['pct_change']:7.1f}%")

# Visualization: Normal vs Control Delta FC comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

targets_plot = [d['target'] for d in comparison_data]
normal_delta = [d['normal_delta_mean'] for d in comparison_data]
control_delta = [d['control_delta_mean'] for d in comparison_data]

x = np.arange(len(targets_plot))
width = 0.35

ax = axes[0]
ax.bar(x - width/2, normal_delta, width, label='Normal (STIM_AMP=10)', alpha=0.7, color='orange')
ax.bar(x + width/2, control_delta, width, label='Control (STIM_AMP=0)', alpha=0.7, color='skyblue')
ax.set_xlabel('Target')
ax.set_ylabel('Delta FC (Stim - Rest)')
ax.set_title('Delta FC Comparison: Normal vs Control')
ax.set_xticks(x)
ax.set_xticklabels(targets_plot)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)

# Difference plot
ax = axes[1]
delta_differences = [d['delta_difference'] for d in comparison_data]
colors = ['green' if d > 0 else 'red' for d in delta_differences]
ax.bar(targets_plot, delta_differences, color=colors, alpha=0.7, edgecolor='black')
ax.set_xlabel('Target')
ax.set_ylabel('Delta FC Difference\n(Normal - Control)')
ax.set_title('Amplitude Effect: Delta FC Difference by Target')
ax.grid(True, axis='y', alpha=0.3)
ax.axhline(0, color='black', linestyle='-', linewidth=1)

plt.tight_layout()
plt.savefig('normal_vs_control_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nComparison visualization saved.")

## Step 12: Save Validation Results

In [ ]:
import json

# Compile all validation results
validation_results = {
    'metadata': {
        'simulation_parameters': {
            'burn_in': BURN_IN,
            'noise_sigma': NOISE_SIGMA,
            'stim_amplitude': STIM_AMP,
            'rho_mm': RHO_MM,
            'roi_num': ROI_num,
            'using_steps': using_steps,
            'remove_initial_trs': remove_initial_trs,
            'bandpass_hz': [low_hz, high_hz],
            'tr_rest': tr_rest,
            'tr_stim': tr_stim
        },
        'preprocessing': {
            'filter_order': 2,
            'zscore': True,
            'cortical_rois': '50-449 (Schaefer 400)'
        }
    },
    'session_specific_fc_correlation': {
        'normal_dataset': {
            'rest_fc': {
                'overall_mean': float(np.mean(all_rest_fc_corrs)) if all_rest_fc_corrs else np.nan,
                'overall_std': float(np.std(all_rest_fc_corrs)) if all_rest_fc_corrs else np.nan,
                'n_sessions': len(all_rest_fc_corrs)
            },
            'stim_fc': {
                'overall_mean': float(np.mean(all_stim_fc_corrs)) if all_stim_fc_corrs else np.nan,
                'overall_std': float(np.std(all_stim_fc_corrs)) if all_stim_fc_corrs else np.nan,
                'n_sessions': len(all_stim_fc_corrs)
            }
        },
        'control_dataset': {
            'rest_fc': {
                'overall_mean': float(np.mean(all_rest_fc_corrs_ctrl)) if all_rest_fc_corrs_ctrl else np.nan,
                'overall_std': float(np.std(all_rest_fc_corrs_ctrl)) if all_rest_fc_corrs_ctrl else np.nan,
                'n_sessions': len(all_rest_fc_corrs_ctrl)
            },
            'stim_fc': {
                'overall_mean': float(np.mean(all_stim_fc_corrs_ctrl)) if all_stim_fc_corrs_ctrl else np.nan,
                'overall_std': float(np.std(all_stim_fc_corrs_ctrl)) if all_stim_fc_corrs_ctrl else np.nan,
                'n_sessions': len(all_stim_fc_corrs_ctrl)
            }
        }
    },
    'per_target_analysis': {
        'normal_dataset': {},
        'control_dataset': {},
        'comparison': []
    }
}

# Add per-target results for normal dataset
for target in targets_list:
    validation_results['per_target_analysis']['normal_dataset'][str(target)] = {
        'rest_fc_mean': float(per_target_results[target]['rest_mean']),
        'rest_fc_std': float(per_target_results[target]['rest_std']),
        'stim_fc_mean': float(per_target_results[target]['stim_mean']),
        'stim_fc_std': float(per_target_results[target]['stim_std']),
        'delta_fc_mean': float(per_target_results[target]['delta_mean']),
        'delta_fc_std': float(per_target_results[target]['delta_std']),
        'n_subjects': int(per_target_results[target]['n_subjects'])
    }

# Add per-target results for control dataset
for target in targets_list:
    validation_results['per_target_analysis']['control_dataset'][str(target)] = {
        'rest_fc_mean': float(per_target_results_ctrl[target]['rest_mean']),
        'rest_fc_std': float(per_target_results_ctrl[target]['rest_std']),
        'stim_fc_mean': float(per_target_results_ctrl[target]['stim_mean']),
        'stim_fc_std': float(per_target_results_ctrl[target]['stim_std']),
        'delta_fc_mean': float(per_target_results_ctrl[target]['delta_mean']),
        'delta_fc_std': float(per_target_results_ctrl[target]['delta_std']),
        'n_subjects': int(per_target_results_ctrl[target]['n_subjects'])
    }

# Add comparison data
for row in comparison_data:
    validation_results['per_target_analysis']['comparison'].append({
        'target': int(row['target']),
        'normal_delta_fc': float(row['normal_delta_mean']),
        'control_delta_fc': float(row['control_delta_mean']),
        'amplitude_effect_delta_fc': float(row['delta_difference']),
        'amplitude_effect_pct_change': float(row['pct_change'])
    })

# Save to JSON
output_file = 'ANN2_validation_results.json'
with open(output_file, 'w') as f:
    json.dump(validation_results, f, indent=2)

print(f"\nValidation results saved to: {output_file}")
print(f"\nSummary:")
print(f"  Simulation parameters: {STIM_AMP=}, {NOISE_SIGMA=}, {BURN_IN=}, {RHO_MM=}")
print(f"  Session-Specific FC (Normal): Rest={np.mean(all_rest_fc_corrs):.4f}, Stim={np.mean(all_stim_fc_corrs):.4f}")
print(f"  Session-Specific FC (Control): Rest={np.mean(all_rest_fc_corrs_ctrl):.4f}, Stim={np.mean(all_stim_fc_corrs_ctrl):.4f}")
print(f"  Per-target analysis: {len(targets_list)} targets analyzed")
print(f"  Comparison: Amplitude effect computed for all targets")

## Step 13: Interpretation of Results

### Session-Specific FC Correlation (Metric 1)
This metric measures how well the simulated functional connectivity (FC) matches the empirical FC for each fMRI session in the cortical regions (Schaefer 400 ROIs). A correlation close to 1.0 indicates that the ANN model accurately captures the session-specific FC patterns.

- **Normal Dataset (STIM_AMP=10)**: Expected high FC correlation, indicating good match between empirical and simulated networks during both rest and stimulation.
- **Control Dataset (STIM_AMP=0)**: Provides a baseline comparison. Since no amplitude modulation is applied, this shows the model's ability to capture brain states without stimulation effects.

### Per-Target FC Analysis (Metrics 2-4)
For each stimulation target, we compute three metrics:

- **Rest FC Correlation**: FC correlation during rest periods (baseline connectivity)
- **Stim FC Correlation**: FC correlation during stimulation periods
- **Delta FC**: The difference between Stim FC and Rest FC, indicating stimulation-induced connectivity changes

### Comparison: Amplitude Effect (Normal vs Control)
The key innovation of this v2 analysis:

- **Normal Dataset**: Full STIM_AMP=10 effect applied during stimulation
- **Control Dataset**: STIM_AMP=0 with REST initialization - shows what happens without amplitude modulation

**The Amplitude Effect** = Delta FC (Normal) - Delta FC (Control)

This shows where the ANN model produces amplitude-dependent connectivity changes:
- Positive values: Targets where higher stimulation amplitude increases connectivity changes
- Negative values: Targets where higher stimulation amplitude decreases connectivity changes
- Values near zero: Targets where amplitude has minimal effect on connectivity patterns

### Design Innovation: REST Initialization
In the control dataset, both REST and STIM conditions are initialized from the same empirical REST data. This ensures that any differences between normal and control datasets come purely from the STIM_AMP parameter, not from different initialization states.